In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import sys
if 'google.colab' in sys.modules:
    %pip install -q tensorflow scikit-learn pytorch einops

# All imports
import os
import sys
import math
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from typing import List, Sequence, Tuple, Optional, Union

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import utils as np_utils
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense, Activation, Dropout, Conv2D, AveragePooling2D, MaxPooling2D,
    SeparableConv2D, DepthwiseConv2D, BatchNormalization, SpatialDropout2D,
    Input, Flatten, GlobalMaxPooling2D
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:

tf.config.list_physical_devices('GPU')


In [ ]:

# Paths & basic config (edit BASE_DIR if running on Colab)
BASE_DIR = '/content/drive/MyDrive/'
LABELED_EEG_DATA_DIR = Path(BASE_DIR) / 'labeled_eeg_data'
PRE_WINDOW_MS = 500
POST_WINDOW_MS = 1000

# --------------------------
# CHANNEL CONFIGURATION
# --------------------------
DEFAULT_EEGNET_CHANNELS = ['F3', 'F4', 'F7', 'F8', 'Fz', 'FC1', 'FC2', 'FC5', 'FC6',
                          'Fp1', 'Fp2', 'Pz', 'P3', 'P4', 'P7', 'P8']

# Auto-discover subjects
USE_SUBJECTS = sorted([
    d.name for d in Path(LABELED_EEG_DATA_DIR).iterdir()
    if d.is_dir() and d.name.startswith('P')
])
print(f"Found {len(USE_SUBJECTS)} subjects: {USE_SUBJECTS[:10]} ...")

CHANNELS_TO_USE = DEFAULT_EEGNET_CHANNELS
print(f"Using {len(CHANNELS_TO_USE)} channels: {CHANNELS_TO_USE}")



In [ ]:
from collections import Counter

# --------------------------
# EEG DATA LOADING
# --------------------------
def load_eeg_data(subjects, channels, pre_ms, post_ms, epoch_start_ms=-200, epoch_end_ms=600, fs=500):
    """
    Load EEG data for specified subjects and channels.
    
    Data is loaded from [-500, 1000] ms window, then a specific epoch is extracted.
    
    Args:
        subjects: list of subject IDs
        channels: list of channel names
        pre_ms: pre-fixation window in file (e.g., 500)
        post_ms: post-fixation window in file (e.g., 1000)
        epoch_start_ms: epoch start relative to fixation (default: -200)
        epoch_end_ms: epoch end relative to fixation (default: 600)
        fs: sampling frequency in Hz (default: 500)
    """
    all_X, all_y, all_subjects = [], [], []
    
    # Calculate total time points in loaded data
    total_time_points = int((pre_ms + post_ms) / 1000 * fs)
    fixation_idx = int(pre_ms / 1000 * fs)  # Index of fixation point
    
    # Calculate epoch indices
    epoch_start_idx = fixation_idx + int(epoch_start_ms / 1000 * fs)
    epoch_end_idx = fixation_idx + int(epoch_end_ms / 1000 * fs)
    
    print(f"Loading data from [-{pre_ms}, {post_ms}] ms window")
    print(f"Extracting epoch [{epoch_start_ms}, {epoch_end_ms}] ms (indices {epoch_start_idx}:{epoch_end_idx})")

    for subject in subjects:
        file_path = LABELED_EEG_DATA_DIR / subject / f"{pre_ms}_{post_ms}_training_data.npz"
        if not file_path.exists():
            continue

        # Load with memory mapping first to check channels
        with np.load(file_path, mmap_mode='r') as data:
            ch_names = data['channel_names']

            # Find indices of channels we want
            ch_indices = []
            for ch in channels:
                if ch in ch_names:
                    ch_indices.append(list(ch_names).index(ch))

            if len(ch_indices) == 0:
                continue  # Skip if no channels found

            # Only load the data we need - much more memory efficient
            X_full = data['X']  # Still memory mapped
            y = data['y'][:]    # Load labels (small)

            # Extract specific epoch and channels
            X_selected = X_full[:, ch_indices, epoch_start_idx:epoch_end_idx].copy().astype(np.float32)

        all_X.append(X_selected)
        all_y.append(y)
        all_subjects.extend([subject] * len(y))  # Track which subject each sample belongs to

        # Force garbage collection to free memory
        del X_selected

    # Combine all subjects
    X_combined = np.concatenate(all_X, axis=0).astype(np.float32)
    y_combined = np.concatenate(all_y, axis=0)

    print(f"\nLoaded {X_combined.shape[0]} trials from {len(set(all_subjects))} subjects")
    print(f"Using {len(ch_indices)} channels, {X_combined.shape[2]} time points")
    print(f"Epoch duration: {(epoch_end_ms - epoch_start_ms)} ms")

    # Add label distribution analysis
    label_counts = Counter(y_combined)
    total_samples = len(y_combined)
    print("\nLabel Distribution:")
    for label, count in sorted(label_counts.items()):
        proportion = count / total_samples
        print(f"  Label {label}: {count} samples ({proportion:.2%})")

    return X_combined, y_combined, np.array(all_subjects)



In [ ]:

# --------------------------
# PER-SUBJECT NORMALIZATION
# --------------------------
def normalize_per_subject(X, y, subjects):
    """
    Normalize each subject's data individually using StandardScaler.
    
    Args:
        X: (samples, channels, time_points)
        y: (samples,)
        subjects: (samples,) array of subject IDs
    
    Returns:
        X_normalized: normalized data
        scalers: dict of StandardScaler objects per subject
    """
    X_normalized = X.copy()
    scalers = {}
    
    unique_subjects = np.unique(subjects)
    for subject in unique_subjects:
        mask = subjects == subject
        X_subject = X[mask]
        
        # Fit scaler on this subject's data
        scaler = StandardScaler()
        X_reshaped = X_subject.reshape(X_subject.shape[0], -1)
        scaler.fit(X_reshaped)
        
        # Transform this subject's data
        X_normalized[mask] = scaler.transform(X_reshaped).reshape(X_subject.shape)
        scalers[subject] = scaler
        
        print(f"  {subject}: mean={scaler.mean_.mean():.4f}, std={scaler.scale_.mean():.4f}")
    
    return X_normalized, scalers




In [ ]:

# --------------------------
# METHOD 1: NORMAL RANDOM SPLIT WITH PER-SUBJECT NORMALIZATION
# --------------------------
def load_data_normal_split(test_size=0.2, val_size=0.2, seed=42):
    """
    Normal random train/val/test split with per-subject normalization.
    This method mixes subjects but normalizes each subject individually before splitting.
    """
    print("\n" + "="*70)
    print("LOADING DATA: NORMAL RANDOM SPLIT (per-subject normalization)")
    print("="*70)
    
    # Load all data with epoch extraction
    X_all, y_all, subjects_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS,
                                               epoch_start_ms=-200, epoch_end_ms=600, fs=500)
    
    # Normalize each subject individually
    print("\nNormalizing per subject:")
    X_normalized, scalers = normalize_per_subject(X_all, y_all, subjects_all)
    
    # Add EEGNet dimension
    X_normalized = X_normalized[..., np.newaxis]  # (N, Chans, Time, 1)
    
    # Random train/val/test split (mixes subjects)
    print("\nPerforming random train/val/test split...")
    X_train, X_test, y_train, y_test, subj_train, subj_test = train_test_split(
        X_normalized, y_all, subjects_all, test_size=test_size, random_state=seed, stratify=y_all
    )
    
    X_train, X_val, y_train, y_val, subj_train, subj_val = train_test_split(
        X_train, y_train, subj_train, test_size=val_size/(1-test_size), random_state=seed, stratify=y_train
    )
    
    # One-hot encode labels
    y_train_cat = tf.keras.utils.to_categorical(y_train.astype(int), 2)
    y_val_cat = tf.keras.utils.to_categorical(y_val.astype(int), 2)
    y_test_cat = tf.keras.utils.to_categorical(y_test.astype(int), 2)
    
    print(f"\nData shapes:")
    print(f"  Train: {X_train.shape} from {len(np.unique(subj_train))} subjects")
    print(f"  Val:   {X_val.shape} from {len(np.unique(subj_val))} subjects")
    print(f"  Test:  {X_test.shape} from {len(np.unique(subj_test))} subjects")
    
    return X_train, X_val, X_test, y_train_cat, y_val_cat, y_test_cat




In [ ]:

# --------------------------
# METHOD 2: LEAVE-ONE-OUT CROSS-VALIDATION (N subjects left out)
# --------------------------
def load_data_loocv(n_test_subjects=3, seed=42):
    """
    Leave-one-out style CV: N subjects are held out for testing, 
    remaining subjects are split into train/val (80/20).
    Normalization is done per-subject before any splitting.
    
    Args:
        n_test_subjects: number of subjects to hold out for testing
        seed: random seed
    
    Returns:
        List of tuples (X_train, X_val, X_test, y_train, y_val, y_test, test_subjects)
        One fold per iteration, cycling through different subject combinations.
    """
    print("\n" + "="*70)
    print(f"LOADING DATA: LEAVE-{n_test_subjects}-OUT CV (per-subject normalization)")
    print("="*70)
    
    # Load all data with epoch extraction
    X_all, y_all, subjects_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS,
                                               epoch_start_ms=-200, epoch_end_ms=600, fs=500)
    
    # Normalize each subject individually
    print("\nNormalizing per subject:")
    X_normalized, scalers = normalize_per_subject(X_all, y_all, subjects_all)
    
    # Add EEGNet dimension
    X_normalized = X_normalized[..., np.newaxis]  # (N, Chans, Time, 1)
    
    # Create folds
    unique_subjects = np.unique(subjects_all)
    n_subjects = len(unique_subjects)
    
    print(f"\nCreating folds with {n_test_subjects} test subjects each...")
    print(f"Total subjects: {n_subjects}")
    
    folds = []
    np.random.seed(seed)
    
    # Shuffle subjects and create multiple folds
    shuffled_subjects = np.random.permutation(unique_subjects)
    
    for fold_idx in range(n_subjects // n_test_subjects):
        # Select test subjects
        test_subject_indices = np.arange(fold_idx * n_test_subjects, 
                                         (fold_idx + 1) * n_test_subjects)
        test_subjects = shuffled_subjects[test_subject_indices]
        train_val_subjects = np.setdiff1d(shuffled_subjects, test_subjects)
        
        # Split data by subject
        test_mask = np.isin(subjects_all, test_subjects)
        train_val_mask = np.isin(subjects_all, train_val_subjects)
        
        X_test = X_normalized[test_mask]
        y_test = y_all[test_mask]
        
        X_train_val = X_normalized[train_val_mask]
        y_train_val = y_all[train_val_mask]
        subj_train_val = subjects_all[train_val_mask]
        
        # Further split train_val into train/val (80/20)
        X_train, X_val, y_train, y_val, subj_train, subj_val = train_test_split(
            X_train_val, y_train_val, subj_train_val, 
            test_size=0.2, random_state=seed, stratify=y_train_val
        )
        
        # One-hot encode
        y_train_cat = tf.keras.utils.to_categorical(y_train.astype(int), 2)
        y_val_cat = tf.keras.utils.to_categorical(y_val.astype(int), 2)
        y_test_cat = tf.keras.utils.to_categorical(y_test.astype(int), 2)
        
        print(f"\nFold {fold_idx + 1}:")
        print(f"  Test subjects: {test_subjects}")
        print(f"  Train: {X_train.shape} from {len(np.unique(subj_train))} subjects")
        print(f"  Val:   {X_val.shape} from {len(np.unique(subj_val))} subjects")
        print(f"  Test:  {X_test.shape}")
        
        folds.append({
            'X_train': X_train,
            'X_val': X_val,
            'X_test': X_test,
            'y_train': y_train_cat,
            'y_val': y_val_cat,
            'y_test': y_test_cat,
            'test_subjects': test_subjects,
            'fold_idx': fold_idx
        })
    
    return folds

In [ ]:
import torch
import torch.nn as nn
from models.cbramod import CBraMod
from einops.layers.torch import Rearrange

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = CBraMod().to(device)
model.load_state_dict(torch.load('pretrained_weights/pretrained_weights.pth', map_location=device))
model.proj_out = nn.Identity()
classifier = nn.Sequential(
  Rearrange('b c s p -> b (c s p)'),
  nn.Linear(22*4*200, 4*200),
  nn.ELU(),
  nn.Dropout(0.1),
  nn.Linear(4 * 200, 200),
  nn.ELU(),
  nn.Dropout(0.1),
  nn.Linear(200, 4),
).to(device)

# mock_eeg.shape = (batch_size, num_of_channels, time_segments, points_per_patch)
mock_eeg = torch.randn((8, 22, 4, 200)).to(device)

# logits.shape = (batch_size, num_of_classes)
logits = classifier(model(mock_eeg))